# SolarSense EMS — Panel Defect Classification with EfficientNet-B2
**Module:** Solar Panel Condition & Fault Classification (Computer Vision Signal)  
**Lead Engineer:** Zaeem  
**Repository:** [Abubakar651/SolarSense-Ems](https://github.com/Abubakar651/SolarSense-Ems)  

---

### Overview
This notebook fine-tunes an ImageNet-pretrained **PyTorch EfficientNet-B2** on the curated, leak-free **SolarSense Hybrid V1 / Master V1** dataset. The model classifies solar panel surface images into 5 operational categories:
1.  — Pristine photovoltaic panels.
2.  — Uniform environmental dust & atmospheric particulate deposition.
3.  — Localized bird dropping shading/hotspot risks.
4.  — Delamination, burned cells, hot spots.
5.  — Micro-cracks, shatter, structural breaks.

The pipeline monitors **Validation Macro-F1** with Cosine Annealing learning rate schedule and evaluates holdout test set performance against the targets:
- **Test Accuracy:** $\ge 94.0\%$
- **Test Macro-F1:** $\ge 0.920$


In [ ]:
# 1. Verify GPU Acceleration
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Please go to Runtime -> Change runtime type -> Select T4 GPU or A100.")


In [ ]:
# 2. Clone Repository & Navigate
import os
if not os.path.exists("SolarSense-Ems"):
    !git clone https://github.com/Abubakar651/SolarSense-Ems.git
%cd SolarSense-Ems
!git status


In [ ]:
# 3. Install Required Dependencies
!pip install -q scikit-learn seaborn matplotlib torchvision pillow


### 4. Dataset Setup (Option A or Option B)

**Option A (Recommended & Fastest):** Upload  (~54 MB) or  to your Colab session or Google Drive, then unzip below.

**Option B:** Automatically download the raw Kaggle datasets using your  API token and run the Protocol 1 deduplication engine.


In [ ]:
# OPTION A: One-Click Download from Google Drive (Recommended)
# Downloads solarsense_processed_panel_images.zip directly to Colab
!pip install -q gdown
!gdown --id 15S6Ja9c2nQhYoZaGcggXjGBgyF2L1mSL -O solarsense_processed_panel_images.zip
# Unzip (note: the zip already contains the folder path data/processed/panel_images/)
!unzip -q -o solarsense_processed_panel_images.zip


In [ ]:
# OPTION B: Automated Kaggle API Ingestion & Protocol 1 Curation
# Uncomment and run if you uploaded kaggle.json to /content/kaggle.json
# !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !python3 src/vision/download_datasets.py
# !python3 src/vision/build_hybrid_dataset.py


In [ ]:
# 5. Protocol 1 Dataset Integrity & Leakage Verification
!python3 src/vision/verify_dataset_a.py


In [ ]:
# 6. Train the EfficientNet-B2 Defect Classifier
from pathlib import Path

# Automatically discover the dataset split root anywhere in the workspace
train_dirs = [p for p in Path(".").rglob("train") if p.is_dir() and any(p.iterdir())]
if not train_dirs:
    train_dirs = [p for p in Path("/content").rglob("train") if p.is_dir() and any(p.iterdir())]

if not train_dirs:
    raise FileNotFoundError("Could not find train/ folder. Please check if the unzip step ran.")

dataset_root = str(train_dirs[0].parent)
classes = sorted([d.name for d in train_dirs[0].iterdir() if d.is_dir()])
print(f"[DATASET] Successfully discovered dataset root: {dataset_root}")
print(f"[CLASSES] Found {len(classes)} classes: {classes}")

# Fine-tunes ImageNet pretrained weights with Adam (lr=3e-4) and Cosine Annealing
!python3 src/vision/train_classifier.py --root "{dataset_root}" --epochs 15 --batch-size 32 --lr 0.0003


In [ ]:
# 7. In-Depth Accuracy & Confusion Matrix Analysis
from src.vision.evaluate_classifier import evaluate_model
from pathlib import Path

train_dirs = [p for p in Path(".").rglob("train") if p.is_dir() and any(p.iterdir())]
if not train_dirs:
    train_dirs = [p for p in Path("/content").rglob("train") if p.is_dir() and any(p.iterdir())]
dataset_root = str(train_dirs[0].parent)

results = evaluate_model(
    checkpoint_path="models/efficientnet_b2_defects.pth",
    dataset_root=dataset_root,
    batch_size=32,
    save_plot_path="docs/samples/panel_defect_confusion_matrix.png"
)


In [ ]:
# 8. Display Confusion Matrix Visualization
from IPython.display import Image, display
display(Image("docs/samples/panel_defect_confusion_matrix.png"))


In [ ]:
# 9. Interactive Visualizer: Test Predictions on Random Panel Images
import random
from PIL import Image as PILImage
import matplotlib.pyplot as plt
from torchvision import transforms
from src.vision.train_classifier import build_model
from src.vision.dataloader import IMAGE_SIZE, IMAGENET_MEAN, IMAGENET_STD
from pathlib import Path
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load("models/efficientnet_b2_defects.pth", map_location=device)
class_to_idx = checkpoint["class_to_idx"]
idx_to_class = {v: k for k, v in class_to_idx.items()}

model = build_model(num_classes=len(class_to_idx), pretrained=False).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Find sample test images
train_dirs = [p for p in Path(".").rglob("train") if p.is_dir() and any(p.iterdir())]
if not train_dirs:
    train_dirs = [p for p in Path("/content").rglob("train") if p.is_dir() and any(p.iterdir())]
dataset_root = train_dirs[0].parent
test_root = dataset_root / "test"
test_imgs = list(test_root.rglob("*.jpg")) + list(test_root.rglob("*.png"))

if test_imgs:
    sample_imgs = random.sample(test_imgs, min(5, len(test_imgs)))
    fig, axes = plt.subplots(1, len(sample_imgs), figsize=(18, 4))
    if len(sample_imgs) == 1: axes = [axes]
    
    for ax, p in zip(axes, sample_imgs):
        raw_img = PILImage.open(p).convert("RGB")
        tensor = transform(raw_img).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(tensor)
            probs = torch.softmax(logits, dim=1)[0]
            pred_idx = logits.argmax(dim=1).item()
            pred_label = idx_to_class[pred_idx]
            conf = probs[pred_idx].item() * 100

        true_label = p.parent.name
        color = "green" if pred_label == true_label else "red"
        ax.imshow(raw_img)
        ax.set_title(f"True: {true_label}
Pred: {pred_label} ({conf:.1f}%)", color=color, fontsize=11, fontweight='bold')
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No test images found to display.")
